## Implement model stacking using multiple classifiers without using built-in stacking functions, manually designing meta-models, optimizing base learners, and handling data leakage issues.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('heart.csv')                        
X, y = df.drop(columns='target').values.astype(float), df['target'].values
print(X.shape, '| positive share:', y.mean().round(2))

(303, 13) | positive share: 0.54


In [2]:
sig = lambda z: 1 / (1 + np.exp(-z))
def logit(p): p = np.clip(p, 0.01, 0.99); return np.log(p / (1 - p))

def auc(t, s):
    sp, sn = s[t == 1], s[t == 0]
    return (sp[:, None] > sn[None, :]).mean() + 0.5 * (sp[:, None] == sn[None, :]).mean()

def logloss(t, s):
    p = np.clip(sig(s), 1e-6, 1 - 1e-6); return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))

def metrics(t, s):
    return dict(acc=np.mean((s > 0) == t), auc=auc(t, s), logloss=logloss(t, s))

In [3]:
def fit_lr(A, t, C):
    m = LogisticRegression(C=C, max_iter=1000).fit(A, t); return lambda Z: m.decision_function(Z)
def fit_knn(A, t, k):
    m = KNeighborsClassifier(k).fit(A, t); return lambda Z: logit(m.predict_proba(Z)[:, 1])
def fit_svm(A, t, C, gamma):
    m = SVC(C=C, gamma=gamma).fit(A, t); return lambda Z: m.decision_function(Z)
def fit_rf(A, t, max_depth, min_samples_leaf):
    m = RandomForestClassifier(100, max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=0).fit(A, t)
    return lambda Z: logit(m.predict_proba(Z)[:, 1])

learners = {
    'LR':  (fit_lr,  [dict(C=c) for c in [0.01, 0.1, 1, 10]], dict(C=1.0)),
    'kNN': (fit_knn, [dict(k=k) for k in [3, 5, 9, 15, 21]], dict(k=5)),
    'SVM': (fit_svm, [dict(C=c, gamma=g) for c in [0.3, 1, 3] for g in [0.01, 0.03, 0.1]], dict(C=1.0, gamma='scale')),
    'RF':  (fit_rf,  [dict(max_depth=d, min_samples_leaf=l) for d in [3, 5, None] for l in [1, 3]], dict(max_depth=None, min_samples_leaf=1)),
}

In [4]:
def make_folds(n, k, seed):
    return np.array_split(np.random.default_rng(seed).permutation(n), k)

def oof_scores(fit, params, A, t, seed, k=5):
    z = np.zeros(len(t))
    for f in make_folds(len(t), k, seed):
        tr = np.setdiff1d(np.arange(len(t)), f)
        z[f] = fit(A[tr], t[tr], **params)(A[f])          
    return z

def tune(fit, grid, A, t, seed):
    best = None
    for params in grid:
        z = oof_scores(fit, params, A, t, seed); loss = logloss(t, z)
        if best is None or loss < best[0]: best = (loss, params, z)
    return best[1], best[2]                                 

In [5]:
def fit_meta(Z, t, lam=0.05, lr=0.1, iters=1000):
    mu, sd = Z.mean(0), Z.std(0) + 1e-9; Zs = (Z - mu) / sd         
    w, b = np.zeros(Z.shape[1]), 0.0
    for _ in range(iters):
        p = sig(Zs @ w + b)
        w -= lr * (Zs.T @ (p - t) / len(t) + lam * w); b -= lr * np.mean(p - t)
    scorer = lambda Zn: ((Zn - mu) / sd) @ w + b
    scorer.w = w                                                   
    return scorer

In [6]:
names = list(learners)
results, meta_w, leaky_w, corrs, acc_in, acc_oof = {}, [], [], [], [], []
add = lambda key, m: results.setdefault(key, []).append(m)

for rep in range(2):
    folds = make_folds(len(X), 5, rep)
    for k in range(5):
        te = folds[k]; tr = np.concatenate([folds[i] for i in range(5) if i != k]); seed = rep * 5 + k
        mu, sd = X[tr].mean(0), X[tr].std(0); A, B = (X[tr] - mu) / sd, (X[te] - mu) / sd  
        t, u = y[tr], y[te]
        Z_oof, Z_te, Z_in = [], [], []
        for name, (fit, grid, default) in learners.items():
            best, oof = tune(fit, grid, A, t, seed)                                       
            final = fit(A, t, **best)                                                     
            add(f'{name} (default)', metrics(u, fit(A, t, **default)(B)))
            add(f'{name} (tuned)', metrics(u, final(B)))
            Z_oof.append(oof); Z_te.append(final(B)); Z_in.append(final(A))               
        Z_oof, Z_te, Z_in = np.c_[tuple(Z_oof)], np.c_[tuple(Z_te)], np.c_[tuple(Z_in)]
        acc_in.append(np.mean((Z_in > 0) == t[:, None], 0)); acc_oof.append(np.mean((Z_oof > 0) == t[:, None], 0))   
        meta, leaky = fit_meta(Z_oof, t), fit_meta(Z_in, t)
        add('Simple average', metrics(u, Z_te.mean(1)))
        add('Stacking (OOF meta-model)', metrics(u, meta(Z_te)))
        add('Stacking (LEAKY: in-sample)', metrics(u, leaky(Z_te)))
        meta_w.append(meta.w); leaky_w.append(leaky.w); corrs.append(np.corrcoef(Z_oof.T))
print('Outer folds:', len(results['Simple average']))

Outer folds: 10


In [7]:
base = np.array([m['acc'] for m in results['LR (tuned)']])
rows = []
for key, ms in results.items():
    g = lambda k: np.array([m[k] for m in ms]); d = g('acc') - base
    rows.append((key, f"{g('acc').mean():.3f} +- {g('acc').std():.3f}", f"{g('auc').mean():.3f}", f"{g('logloss').mean():.3f}",
                 f"{d.mean():+.3f} +- {d.std(ddof=1) / np.sqrt(len(d)):.3f}"))
pd.DataFrame(rows, columns=['Model', 'Accuracy', 'AUC', 'Log loss', 'Acc diff vs LR (tuned)'])

,Model,Accuracy,AUC,Log loss,Acc diff vs LR (tuned)
0,LR (default),0.817 +- 0.051,0.906,0.407,-0.008 +- 0.006
1,LR (tuned),0.825 +- 0.048,0.908,0.398,+0.000 +- 0.000
2,kNN (default),0.830 +- 0.069,0.884,0.482,+0.005 +- 0.012
3,kNN (tuned),0.828 +- 0.078,0.909,0.402,+0.003 +- 0.012
4,SVM (default),0.822 +- 0.057,0.902,0.466,-0.003 +- 0.009
5,SVM (tuned),0.832 +- 0.058,0.907,0.428,+0.007 +- 0.005
6,RF (default),0.807 +- 0.045,0.901,0.414,-0.018 +- 0.012
7,RF (tuned),0.813 +- 0.056,0.903,0.422,-0.012 +- 0.015
8,Simple average,0.837 +- 0.069,0.914,0.399,+0.012 +- 0.010
9,Stacking (OOF meta-model),0.828 +- 0.062,0.913,0.391,+0.003 +- 0.009


In [8]:
print('Base learners ka accuracy: TRAIN rows pe in-sample vs out-of-fold (wahi rows, par model ne unhe nahi dekha)')
print(f"{'':14s} " + ' '.join(f'{n:>8s}' for n in names))
print(f"{'In-sample':14s} " + ' '.join(f'{v:8.3f}' for v in np.mean(acc_in, 0)))
print(f"{'Out-of-fold':14s} " + ' '.join(f'{v:8.3f}' for v in np.mean(acc_oof, 0)))
print('\nMeta-model ke weights (standardized scores pe):')
print(f"{'Meta-model':14s} " + ' '.join(f'{n:>8s}' for n in names))
print(f"{'OOF (correct)':14s} " + ' '.join(f'{v:8.2f}' for v in np.mean(meta_w, 0)))
print(f"{'In-sample':14s} " + ' '.join(f'{v:8.2f}' for v in np.mean(leaky_w, 0)))
print('\nBase learners ke OOF scores ki average correlation (kam correlation = zyada diversity):')
print(pd.DataFrame(np.mean(corrs, 0), index=names, columns=names).round(2))

Base learners ka accuracy: TRAIN rows pe in-sample vs out-of-fold (wahi rows, par model ne unhe nahi dekha)
                     LR      kNN      SVM       RF
In-sample         0.853    0.839    0.864    0.949
Out-of-fold       0.830    0.821    0.828    0.828

Meta-model ke weights (standardized scores pe):
Meta-model           LR      kNN      SVM       RF
OOF (correct)      0.49     0.47     0.50     0.60
In-sample          0.35     0.34     0.51     1.38

Base learners ke OOF scores ki average correlation (kam correlation = zyada diversity):
       LR   kNN   SVM    RF
LR   1.00  0.93  0.99  0.91
kNN  0.93  1.00  0.93  0.90
SVM  0.99  0.93  1.00  0.92
RF   0.91  0.90  0.92  1.00
